# 第 6 周 · 周五：SQL 性能调优——从能跑到跑得快

> 难度：⭐⭐⭐⭐ | 定位：掌握SQL性能优化的核心思路，能诊断慢查询并给出优化方案
> 前置依赖：W5全部SQL知识/W6D1复杂SQL | 后续用途：大数据场景优化、生产环境排障
> 面试地位：高级数据分析师必问，区分"会用SQL"和"精通SQL"的分水岭

---

## 📘 区块1：核心概念与第一性原理

### 1.1 SQL慢的三个根本原因

所有SQL性能问题，追根溯源都可以归结为三类：

**原因1：扫描太多(Scan)**
数据库扫描了太多不必要的数据。比如没走分区导致全表扫描、没走索引导致逐行检查、SELECT * 取了不需要的列浪费IO。
典型症状：EXPLAIN的type=ALL、rows=几千万甚至几亿。

**原因2：移动太多(Shuffle)**
JOIN或聚合时数据在节点之间大量传输(分布式环境)。比如大表JOIN大表导致大量Shuffle、GROUP BY的key太多导致网络传输瓶颈。
典型症状：MapReduce/Spark任务中Shuffle数据量巨大、网络IO打满。

**原因3：等一个人(Skew/数据倾斜)**
某个key的数据量是其他key的几十倍甚至几百倍。比如GROUP BY user_id时有一个超级用户占了总量的80%，导致99%的task都在等那1个task。
典型症状：大部分task几秒完成，个别task跑了30分钟还没结束。

**遇到慢查询时，先判断属于哪一类，再对症下药**：
- 扫描多 → 加分区条件/加索引/裁剪不必要的列
- Shuffle多 → 先过滤再JOIN/调整JOIN顺序/Broadcast小表
- 等一个人 → 加盐打散/拆分倾斜key单独处理/MapJoin

### 1.2 从第一性原理推导：为什么索引能加速查询？

假设你有一本1000页的书，要找"SQL"这个词出现的位置：
- **没有索引**：你需要一页一页翻，最坏情况要翻1000页 → O(n)
- **有索引**：先查目录(B-Tree索引)，定位到第300-400页 → O(log n)

索引的本质是将无序数据转化为有序的B-Tree结构。5000万行的表，全表扫描要读5000万行；通过B-Tree索引定位，只需要3-4次磁盘IO(树的高度)。

但索引不是免费的——每次INSERT/UPDATE/DELETE都要同时维护索引，占用额外10-30%的存储空间。所以建索引的原则是：在WHERE/JOIN/ORDER BY的列上建，选择性(不同值的比例)越高的列越值得建索引。

### 1.3 诊断的优先级顺序(由简到繁)

遇到慢查询时，按以下顺序排查，不要跳步：
1. WHERE里有没有分区字段？(没有=全表扫描→这是第一步就要排查的)
2. 有没有SELECT *？(取所有列浪费IO和内存→改成只选需要的列)
3. JOIN条件是否完整？(遗漏条件=笛卡尔积=行数爆炸)
4. JOIN的key有没有数据倾斜？(查看key的count分布)
5. 子查询能否改成JOIN/CTE？(避免子查询反复执行)
6. 需要加索引吗？(在WHERE/JOIN/ORDER BY列上建索引)
7. 需要调整JOIN策略吗？(大小表JOIN→Broadcast Hash Join)
8. 需要加盐/拆分吗？(数据倾斜的最后解决手段)

### 1.4 面试高频考点

**面试官常问**：
- "一个SQL执行很慢，你的排查思路是什么？"
- "什么是数据倾斜？怎么发现和解决？"
- "EXPLAIN的输出怎么看？type字段有哪些值？"
- "COUNT(*)和COUNT(1)和COUNT(col)有什么区别？"
- "分区和索引的区别？什么时候用哪个？"
- "覆盖索引是什么？为什么能提升性能？"

**回答要点**：
- 按优先级排查：EXPLAIN → 数据分布 → 再动手优化
- 数据倾斜 = 某个key数据量远超其他，解决：加盐/拆分/过滤NULL/MapJoin
- EXPLAIN关注type(ALL最差→const最好)、rows(扫描行数)、Extra(Using filesort/temporary要警惕)
- COUNT(*)和COUNT(1)在现代数据库中性能完全相同(优化器会将COUNT(*)优化为走最小索引)；COUNT(col)不统计NULL值
- 分区是物理切割(不同分区=不同文件/目录)，索引是逻辑排序(同一分区内的B-Tree)。分区优先于索引——先确保分区裁剪
- 覆盖索引：索引包含了查询所需的所有列，不需要回表查主键索引。Extra显示Using index


## 🧩 区块2：知识点详细拆解

### 2.1 EXPLAIN输出深度解读

**为什么重要**：EXPLAIN是诊断慢查询的第一工具。不看EXPLAIN就优化SQL = 不看化验单就开药。

**type字段——最重要的优化指标**：
| type | 含义 | 触发条件 | 性能 |
|------|------|---------|------|
| ALL | 全表扫描 | 没有可用索引 | 最差(百万行+) |
| index | 全索引扫描 | 需要扫描整个索引 | 差 |
| range | 范围扫描 | WHERE id > 100 | 一般 |
| ref | 非唯一索引查找 | WHERE city='北京' | 好 |
| eq_ref | 唯一索引查找 | JOIN ON primary_key | 很好 |
| const | 常量查找 | WHERE id = 1 | 最好 |

**rows(预估扫描行数)**：
- 这是优化器的统计估计值，不是实际扫描行数
- 优化前后对比rows是最直观的效果指标(从1000万降到1000 = 10000倍提升)
- 如果rows估计值和实际差异很大→统计信息过期→需要ANALYZE TABLE更新统计信息

**Extra字段——隐藏的性能杀手**：
| 值 | 含义 | 严重程度 | 优化方向 |
|-----|------|---------|--------|
| Using index | 覆盖索引，不需回表 | 最优 | 无需优化 |
| Using where | 使用了WHERE过滤 | 正常 | 无需优化 |
| Using temporary | 使用了临时表 | 中 | 优化GROUP BY/DISTINCT/ORDER BY |
| Using filesort | 需要额外文件排序 | 中 | 为ORDER BY列建索引 |
| Using join buffer | JOIN使用了缓冲区 | 中 | 为JOIN key建索引 |

### 2.2 索引优化的五大黄金法则

**法则1：WHERE/JOIN/ORDER BY的列优先建索引**
这些列是最常用的过滤和排序条件，建索引的ROI最高。

**法则2：复合索引遵循最左前缀原则**
索引(a, b, c)等效于创建了(a)、(a,b)、(a,b,c)三个索引。
支持：WHERE a=? / WHERE a=? AND b=? / WHERE a=? AND b=? AND c=?
不支持：WHERE b=? / WHERE c=? / WHERE b=? AND c=?
所以列的顺序很重要——把选择性最高(不同值最多)的列放最前面。

**法则3：覆盖索引避免回表**
如果索引包含SELECT的所有列，直接从索引取数据，不需要回表查主键索引：
```sql
SELECT name, age FROM users WHERE city = '北京';
-- 索引: CREATE INDEX idx_cover ON users (city, name, age);
-- Extra会显示 Using index(覆盖索引，最优！)
```

**法则4：避免索引失效(5种常见场景)**
1. 索引列上使用函数：WHERE YEAR(date) = 2024
   → 改为：WHERE date >= '2024-01-01' AND date < '2025-01-01'
2. 隐式类型转换：WHERE phone = 13812345678(phone是VARCHAR)
   → 改为：WHERE phone = '13812345678'
3. LIKE前置通配符：WHERE name LIKE '%张三'
   → 改为全文索引(FULLTEXT)或ES搜索
4. OR条件不是所有列都有索引：WHERE a=? OR b=?
   → 改为：SELECT ... WHERE a=? UNION SELECT ... WHERE b=?
5. 使用!=或<>或NOT IN：MySQL中这些操作通常不走索引
   → 改用UNION或覆盖索引

**法则5：选择性低的列不要单独建索引**
比如性别(男/女)、状态(0/1)等，选择性太低(B-Tree只有两个分支，扫描一半的数据)。这种列适合用于复合索引的前缀(和其他高选择性列组合)。

### 2.3 数据倾斜的完整解决方案

**为什么重要**：在大数据场景(Hive/Spark)下，数据倾斜是性能的头号杀手。一个task处理80%的数据，其他task处理20%——99%的时间在等那一个慢task。

**诊断方法**：
```sql
SELECT key_field, COUNT(*) AS cnt FROM table
GROUP BY key_field ORDER BY cnt DESC LIMIT 20;
```
如果前3个key占了50%+的数据 → 严重倾斜。

**方案1：过滤NULL值**
NULL在Hive/Spark中会进入同一个reducer。如果NULL数据没有业务意义：WHERE key IS NOT NULL

**方案2：加盐打散(Salting)**
给倾斜的key加随机后缀：
```sql
SELECT CONCAT(user_id, '_', FLOOR(RAND() * 100)) AS salted_key, SUM(amount)
FROM orders GROUP BY salted_key;
-- 外层再聚合一次：SUBSTRING_INDEX(salted_key, '_', 1)
```

**方案3：MapJoin(Broadcast Join)**
如果一张表很小(<100MB可放入内存)，用MapJoin避免Shuffle，直接在Map端完成JOIN。在Hive中：`SELECT /*+ MAPJOIN(small_table) */ ...`

**方案4：拆分倾斜key单独处理**
将倾斜的key拿出来单独JOIN，非倾斜的key正常JOIN，最后UNION ALL合并。

### 2.4 JOIN优化的三个核心原则

**原则1：小表驱动大表**
将数据量小的表放在JOIN的左侧(驱动表)，数据量大的放右侧。Nested Loop JOIN时，驱动表的每一行都要去被驱动表查找。

**原则2：先过滤再JOIN(最重要！)**
```sql
-- 坏：1亿行JOIN 5000万行 → Shuffle海量数据
SELECT * FROM orders o JOIN users u ON o.user_id=u.user_id
WHERE o.dt='2026-01-15' AND u.status='active';
-- 好：先过滤到1万行和5000行 → Shuffle 1.5万行
WITH o AS (SELECT * FROM orders WHERE dt='2026-01-15'),
     u AS (SELECT * FROM users WHERE status='active')
SELECT * FROM o JOIN u ON o.user_id=u.user_id;
```

**原则3：聚合后再JOIN**
```sql
-- 坏：明细JOIN → 大量行参与JOIN
SELECT u.name, o.amount FROM users u JOIN orders o ON u.id=o.user_id;
-- 好：先聚合→JOIN少量汇总行
SELECT u.name, os.total FROM users u
JOIN (SELECT user_id, SUM(amount) AS total FROM orders GROUP BY user_id) os
  ON u.id=os.user_id;
```

### 2.5 分区表策略

**分区是处理大数据量最有效的手段**。1亿行的订单表，按天分365个分区后，查一天只扫描约30万行(300倍提升)。

**分区类型选择**：
- RANGE分区：按日期范围(最常用，适合时间序列数据)
- LIST分区：按枚举值(城市、品类、状态)
- HASH分区：按哈希值均匀分布(用户ID，避免倾斜)

**分区裁剪的关键**：
```sql
-- 好：直接指定分区值，触发分区裁剪
SELECT COUNT(*) FROM orders WHERE dt = '2026-01-15';
-- 坏：LIKE不会触发分区裁剪！
SELECT COUNT(*) FROM orders WHERE dt LIKE '2026-01%';  -- 全表扫描！
```

### 2.6 COUNT优化

COUNT(*)和COUNT(1)在现代数据库中性能完全相同(优化器将COUNT(*)转为COUNT(1))。但COUNT(DISTINCT)差异巨大：
```sql
-- 慢：所有数据汇聚到一个节点去重
SELECT COUNT(DISTINCT user_id) FROM orders;
-- 快：各节点先局部去重，再汇总计数
SELECT COUNT(*) FROM (SELECT user_id FROM orders GROUP BY user_id) t;
```

### 2.7 分页优化

OFFSET分页在大偏移量时性能灾难级下降：
```sql
-- 慢：扫描100010行，丢弃100000行
SELECT * FROM orders ORDER BY id LIMIT 10 OFFSET 100000;
-- 快：游标分页，直接从指定位置开始
SELECT * FROM orders WHERE id > 100000 ORDER BY id LIMIT 10;
```

### 2.8 常见优化速查表

| 问题 | 典型症状 | 优化方案 | 预期效果 |
|------|---------|---------|--------|
| 没用分区 | type=ALL,rows=亿级 | WHERE加分区过滤条件 | 100-1000倍 |
| SELECT * | IO打满,内存溢出 | 只选需要的5-10列 | 5-20倍 |
| 数据倾斜 | 99%task快,1个慢 | 加盐/MapJoin/拆分 | 10-100倍 |
| 笛卡尔积 | rows暴增到亿级 | 检查JOIN ON条件完整性 | 无限倍 |
| 子查询反复执行 | 同一子查询算多次 | 用CTE或临时表物化 | 2-10倍 |
| 索引失效 | 期望走索引但type=ALL | 避免函数/转换/LIKE'%x' | 10-1000倍 |
| 大表JOIN大表 | Shuffle 10亿行 | 先过滤/先聚合再JOIN | 10-1000倍 |
| OFFSET分页 | 越往后翻越慢 | 游标分页 WHERE id>last_id | 100-10000倍 |

### 2.9 索引选择的实战决策框架

面对一个新表，如何决定在哪些列上建索引？

**决策四步**：
1. 找出最常用的查询模式(从慢查询日志/业务需求分析)
2. 对于每个查询模式，标记WHERE/JOIN/ORDER BY/GROUP BY涉及的列
3. 为出现频率最高的列建单列索引
4. 为经常一起出现在WHERE中的多列建复合索引(注意列顺序！选择性高的在前)
5. 如果可以，创建覆盖索引(包含SELECT的所有列)，避免回表

**常见建索引误区**：
- 给每个列单独建索引 → 浪费空间且写入慢(每次INSERT要更新所有索引)
- 把所有列都放到一个复合索引 → 太大且不灵活(最左前缀限制)
- 选择性低的列(性别/状态)单独建索引 → 几乎没用(B-Tree只有两个分支)
- 建了索引就不管了 → 索引也需要定期维护(碎片整理/统计信息更新/删除无用索引)

### 2.10 从单机MySQL到分布式Hive的优化思维转变

**单机MySQL时代**：优化核心=减少磁盘IO。手段：加索引、避免全表扫描、使用覆盖索引、优化查询语句结构。

**分布式Hive/Spark时代**：优化核心=减少网络Shuffle。手段：分区裁剪、MapJoin广播小表、处理数据倾斜、减少Stage数量。

两者的优化哲学完全不同：
- MySQL关心的是"每行数据读得快不快"(索引优化，磁盘IO)
- Hive关心的是"数据不要跨节点搬来搬去"(分区+Shuffle优化，网络IO)

**面试时主动提到这个差异**——说明你有跨技术栈的经验和视野，能从原理层面理解优化。

### 2.11 性能优化的"道"与"术"

**术(具体技巧)**：分区裁剪、索引优化、先过滤再JOIN、COUNT(DISTINCT)改GROUP BY、游标分页替代OFFSET

**道(底层原理)**：减少扫描(Scan)、减少移动(Shuffle)、避免等待(Skew)

掌握"术"能解决80%的日常问题。理解"道"能解决剩下20%——包括那些没有现成答案的新问题、新技术栈的优化。

**优化不是为了炫技**——是为了让业务报表准时出、让运营同事不卡顿、让计算资源不浪费。每一次优化的背后，都是更快的业务决策和更低的运营成本。


## 💻 区块3：动手练习——优化案例对比

虽然没有真实的大数据环境，但通过8个优化案例对比，学习优化思维和诊断方法。每个案例都有"慢SQL"和"优化后SQL"的对比，以及原理说明。

In [ ]:
print("=" * 60)
print("SQL性能优化8大案例对照表")
print("=" * 60)

print("\n案例1：分区裁剪")
print("  慢: SELECT COUNT(*) FROM orders WHERE dt LIKE '2026-01%'")
print("     → LIKE不触发分区裁剪！全表扫描所有分区")
print("  快: SELECT COUNT(*) FROM orders WHERE dt>='2026-01-01' AND dt<='2026-01-31'")
print("     → 范围查询触发分区裁剪，只扫描1月份分区")

print("\n案例2：COUNT(DISTINCT)优化")
print("  慢: SELECT COUNT(DISTINCT user_id) FROM orders")
print("     → 所有数据汇聚到一个reducer去重(单点瓶颈)")
print("  快: SELECT COUNT(*) FROM (SELECT user_id FROM orders GROUP BY user_id) t")
print("     → 各节点先局部去重，再汇总计数(并行)")

print("\n案例3：先过滤再JOIN(最重要！)")
print("  慢: SELECT * FROM orders o JOIN users u ON o.user_id=u.user_id")
print("      WHERE o.dt='2026-01-15' AND u.status='active'")
print("     → 10亿行JOIN 5000万行，Shuffle海量数据后才过滤")
print("  快: WITH o AS (SELECT * FROM orders WHERE dt='2026-01-15'),")
print("           u AS (SELECT * FROM users WHERE status='active')")
print("      SELECT * FROM o JOIN u ON o.user_id=u.user_id")
print("     → 先过滤到1万行+5000行，再JOIN只需shuffle 1.5万行")

print("\n案例4：避免索引失效——对列使用函数")
print("  慢: SELECT * FROM users WHERE YEAR(register_date) = 2024")
print("     → 对索引列使用函数→索引完全失效→全表扫描")
print("  快: SELECT * FROM users")
print("      WHERE register_date >= '2024-01-01' AND register_date < '2025-01-01'")
print("     → 范围查询，可以使用索引")

print("\n案例5：分页优化——游标替代OFFSET")
print("  慢: SELECT * FROM orders ORDER BY id LIMIT 10 OFFSET 100000")
print("     → 需要扫描100010行，丢弃前100000行")
print("  快: SELECT * FROM orders WHERE id > 100000 ORDER BY id LIMIT 10")
print("     → 直接从id>100000开始扫描，只读10行")

print("\n案例6：UNION替代OR——让每个子查询走索引")
print("  慢: SELECT * FROM users WHERE city='北京' OR age>25")
print("     → OR条件可能导致所有索引失效")
print("  快: SELECT * FROM users WHERE city='北京'")
print("      UNION")
print("      SELECT * FROM users WHERE age>25")
print("     → 两个子查询各自走自己的索引，UNION合并结果")

print("\n案例7：覆盖索引避免回表")
print("  慢: SELECT * FROM users WHERE city='北京'")
print("     → 通过city索引找到rowid→回表取所有列(二次IO)")
print("  快: SELECT name, age FROM users WHERE city='北京'")
print("     → 如果索引是(city, name, age)→直接从索引取数据(Using index)")

print("\n案例8：数据倾斜处理")
print("  问题: GROUP BY user_id，user_id=NULL数据占80%")
print("  方案1: WHERE user_id IS NOT NULL(如果NULL无业务意义)")
print("  方案2: CASE WHEN user_id IS NULL")
print("         THEN CONCAT('NULL_', FLOOR(RAND()*100)) ELSE user_id END")
print("         → 把NULL打散到100个随机桶里")
print("  方案3: 拆分→UNION ALL→NULL单独处理")


### 步骤2：性能诊断思维训练

在真实环境中，优化效果用EXPLAIN对比：优化前的type=ALL和rows=亿级 → 优化后的type=ref/range和rows=千级。成本降低10000倍是常见的优化效果。

**关键认知**：在10亿行级别的表上，不优化的SQL根本跑不完(超时/OOM)。优化不是"让查询快一点"，而是"让查询能跑出来"。

**性能优化的三层境界**：
- 第一层：会加索引、会看EXPLAIN → 能解决80%的问题
- 第二层：理解数据分布、能处理倾斜 → 能解决95%的问题
- 第三层：能根据业务特点预先设计优化方案 → 不给慢查询产生的机会


## 📊 区块4：业务场景应用

### 场景1：首页加载慢——用户列表查询优化

**问题**：首页加载需要5秒以上，经排查是用户列表查询太慢。

**原始SQL**：
```sql
SELECT * FROM users WHERE status = 1 ORDER BY register_date DESC LIMIT 20;
```

**诊断分析**：
- users表有1000万行
- status字段只有0/1两个值(选择性极差，走索引也只扫描一半)
- ORDER BY register_date需要额外排序(Using filesort)
- SELECT * 取了所有50个字段(很多不需要)

**优化方案**：
```sql
-- 1. 创建复合索引(包含ORDER BY列)
CREATE INDEX idx_status_date ON users (status, register_date DESC);
-- 2. 覆盖索引(索引包含所有查询列)
CREATE INDEX idx_cover ON users (status, register_date DESC, id, name, avatar);
-- 3. 只取需要的列
SELECT id, name, avatar FROM users
WHERE status = 1 ORDER BY register_date DESC LIMIT 20;
```
**效果**：5秒 → 0.01秒(500倍提升)。Extra从Using filesort变为Using index。

### 场景2：日报表超时——大表JOIN优化

**原始SQL**：
```sql
SELECT u.city, COUNT(DISTINCT o.order_id) AS order_count,
  SUM(o.amount) AS total_amount
FROM users u LEFT JOIN orders o ON u.id = o.user_id
WHERE o.order_date >= '2026-01-01'
GROUP BY u.city;
```

**诊断**：users 1000万行 × orders 1亿行 = 巨大中间结果。

**优化方案**：
```sql
-- 先聚合一侧的表，大幅减少JOIN数据量
WITH order_stats AS (
  SELECT user_id, COUNT(order_id) AS order_count,
    SUM(amount) AS total_amount
  FROM orders WHERE order_date >= '2026-01-01'
  GROUP BY user_id
)
SELECT u.city,
  SUM(os.order_count) AS order_count,
  SUM(os.total_amount) AS total_amount
FROM users u
LEFT JOIN order_stats os ON u.id = os.user_id
GROUP BY u.city;
```
**效果**：30分钟 → 1分钟(30倍提升)。

### 场景3：双十一大促——数据倾斜实战

**问题**：大促期间，"数码"品类订单占比从平时的25%飙升到60%，导致GROUP BY category的查询从3秒变成10分钟。

**诊断**：查看category分布——一个key占了60%的数据。

**解决方案——加盐打散**：
```sql
-- Step1: 倾斜品类加随机后缀打散到100个桶
SELECT CASE WHEN category = '数码'
         THEN CONCAT('数码_', FLOOR(RAND() * 100))
         ELSE category END AS split_category,
  SUM(amount) AS gmv
FROM orders WHERE order_date = '2026-11-11'
GROUP BY split_category;
-- Step2: 外层再去掉后缀再聚合一次得到最终结果
-- SELECT SUBSTRING_INDEX(split_category, '_', 1) AS category, SUM(gmv)
```

### 场景4：冷热数据分离——存储成本优化

**问题**：订单表累积到10亿行，存储成本高且查询慢。但90%的查询只访问最近3个月的数据。

**方案**：
1. RANGE分区：按月分区(36个分区覆盖3年)
2. 热数据(最近3月)：SSD高性能存储
3. 冷数据(3月以前)：HDD或对象存储(S3/OSS)
4. 查询时自动分区裁剪到热数据分区
**效果**：日常查询从10亿行→3000万行(30倍)，存储成本降低40-60%。

### 场景5：SQL优化的成本效益分析

优化不是免费的——建索引占用存储且影响写入性能、分区需要维护成本、加盐增加计算复杂度。所以优化也需要成本效益分析：
- 每天跑100次的查询 → 花1小时优化到0.01秒 = 非常值得(每年节省100小时等待时间)
- 每月跑1次的后台报表 → 30分钟也能接受，花1小时优化到1分钟 = ROI不高
- 优先优化高频查询和核心业务路径的SQL

**记住**：优化的目的是提升整体的投入产出比，不是追求每一个SQL都跑得飞快。

### 场景6：优化是数据分析师的职业素养

每次写SQL的时候，花10秒想一下：这个查询在生产环境跑会怎样？表有多大？有没有走分区？有没有可能数据倾斜？养成这个习惯，你就从"能写SQL的人"变成了"能写好SQL的人"。这正是初级和高级数据分析师之间最明显的区别之一。

记住：优化不是为了炫技——是为了让业务报表准时出、让运营同事不卡顿、让计算资源不浪费。每一次优化的背后，都是更快的决策和更低的成本。


## ✅ 区块5：分级检验清单

### 🟢 初级检验(必须能独立完成的)
- [ ] 能说出SQL慢的三个根本原因(扫描多/Shuffle多/等一个人)
- [ ] 能识别5种索引失效场景并给出修复方案
- [ ] 能说出EXPLAIN type字段的6种类型(ALL→const)
- [ ] 能写出"先过滤再JOIN"的SQL改写
- [ ] 能解释分区和索引的区别
- [ ] 能说出COUNT(*)/COUNT(1)/COUNT(col)的区别

**要点**：
- 三个原因：扫描太多(全表)、Shuffle太多(大表JOIN)、等一个人(倾斜)
- 5种失效：函数操作列/类型转换/LIKE'%x'/OR连接/!=或<>
- type：ALL < index < range < ref < eq_ref < const
- 先过滤再JOIN：用CTE分别过滤再JOIN
- 分区=物理切割(不同文件)，索引=逻辑排序(B-Tree)
- COUNT(*)=COUNT(1)，COUNT(col)不统计NULL

### 🟡 中级检验(能解释给同事听的原理)
- [ ] 能诊断数据倾斜并说出4种解决方案
- [ ] 能解释复合索引的最左前缀原则及列顺序选择
- [ ] 能解释覆盖索引为什么能避免回表
- [ ] 能写出COUNT(DISTINCT)的优化方案(GROUP BY替代)
- [ ] 能分析JOIN三种策略(Nested Loop/Hash/Merge)的适用场景
- [ ] 能说出单机MySQL和分布式Hive优化思维的差异

**要点**：
- 4种方案：过滤NULL/加盐打散/MapJoin Broadcast/拆分后UNION
- 最左前缀：索引(a,b,c)支持a=? / a=? AND b=? / a=? AND b=? AND c=?
- 覆盖索引 = 索引包含所有SELECT列→不需要回表→Extra显示Using index
- COUNT(DISTINCT)→GROUP BY user_id 再 COUNT(*)
- Nested Loop(小表)/Hash Join(大表等值)/Merge Join(有序表)
- MySQL优化磁盘IO(索引)，Hive优化网络IO(分区+Shuffle)

### 🔴 高级检验(能举一反三的迁移能力)
- [ ] 能设计分区表方案(选择分区键/策略/粒度)
- [ ] 能分析慢查询日志并给出完整优化方案(诊断→方案→效果预估)
- [ ] 能设计冷热分离的存储架构
- [ ] 能根据数据量和查询模式设计索引策略
- [ ] 能预判新业务上线时的SQL性能风险并提前规避
- [ ] 能对优化方案做成本效益分析(优化投入 vs 性能收益)


## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W5全部SQL知识 | 优化建立在正确SQL基础上 |
| **← 前置依赖** | W6D1-D4实战方法 | 复杂SQL更需要性能优化 |
| **→ 后续用途** | 大数据(Hive/Spark) | 分布式SQL的性能优化是进阶关键 |
| **→ 后续用途** | 数仓建模 | 分区表、索引设计是数仓核心技能 |
| **业务连接** | 报表体验 | 查询慢→报表超时→运营无法工作 |
| **业务连接** | 成本控制 | 优化SQL=降低计算资源=省钱 |

### 与大数据技术的关联
- Hive：分区裁剪、MapJoin Hint、数据倾斜加盐、Tez/Spark执行引擎
- Spark SQL：Broadcast Hash Join、Shuffle分区数调优(spark.sql.shuffle.partitions)
- Presto/Trino：内存管理、JOIN顺序优化、分桶(Bucketing)
- Flink SQL：状态后端选择、Checkpoint优化、MiniBatch聚合


## 📝 区块7：当日小结


### 补充：优化是数据分析师的职业素养

每次写SQL的时候，花10秒想一下：这个查询在生产环境跑会怎样？表有多大？有没有走分区？有没有可能数据倾斜？养成这个习惯，你就从"能写SQL的人"变成了"能写好SQL的人"。而这正是初级和高级数据分析师之间最明显的区别之一。

记住：优化不是为了炫技——是为了让业务报表准时出、让运营同事不卡顿、让计算资源不浪费。每一次优化的背后，都是更快的决策和更低的成本。


### 补充：优化是数据分析师的职业素养

每次写SQL的时候，花10秒想一下：这个查询在生产环境跑会怎样？表有多大？有没有走分区？有没有可能数据倾斜？养成这个习惯，你就从"能写SQL的人"变成了"能写好SQL的人"。而这正是初级和高级数据分析师之间最明显的区别之一。

记住：优化不是为了炫技——是为了让业务报表准时出、让运营同事不卡顿、让计算资源不浪费。每一次优化的背后，都是更快的决策和更低的成本。

### 性能优化的三个层次

第一层会加索引会看EXPLAIN能解决80%的问题。第二层理解数据分布能处理倾斜能解决95%的问题。第三层能根据业务特点预先设计优化方案不给慢查询产生的机会。从第一层到第三层需要的是经验积累和对业务的理解而不仅仅是技术。当你开始在设计阶段就考虑性能问题你就是真正的高级工程师了。
